In [1]:
import yaml
import torch
import pytorch_lightning as pl
from tqdm.auto import tqdm
import torch.nn as nn
import numpy as np 

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import h5py
import numpy as np
import os
import pandas as pd
import pathlib
import pickle
import scipy.stats as stats
import soxr
#! change below to spatial_attn_lighting if want to use with modular 
import src.spatial_attn_lightning as attn_tracking_lightning
import src.audio_transforms as at
import torch
import yaml

import argparse
from argparse import ArgumentParser
from corpus.speaker_room_dataset import SpeakerRoomDataset
from tqdm.auto import tqdm
from datetime import datetime
import sys 
import torchaudio.transforms as T

sys.path.append('../')
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
# torch._dynamo.config.skip_nnmodule_hook_guards=False

In [3]:
config_path = "config/binaural_attn/word_task_half_co_loc_v08_gender_bal_4M_sanity.yaml"
ckpt_path = "attn_cue_models/word_task_half_co_loc_v08_gender_bal_4M_sanity/checkpoints/epoch=7-step=89878.ckpt"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
# config['model']['new_modulee']=Tr
# config['getting_acts']  = True
model = attn_tracking_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=ckpt_path, config=config, strict=True)


Using explicit dim specification for demeaning in audio transforms
Using BinauralAuditoryAttentionCNN
v08 True
num_classes={'num_words': 800}
Model performing word task
Conv block order: LN -> Conv -> ReLU
coch_affine: True


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [4]:
model_name = pathlib.Path(config_path).stem


In [5]:
from corpus.jsinV3_attn_tracking_multi_talker_background import jsinV3_attn_tracking_multi_talker_background
import src.audio_transforms as at

coch_gram = model.coch_gram.cuda()
label_type = 'CV'
sr = config['audio']['rep_kwargs']['sr']
config['data'] = {}
config['data']['corpus'] = {}
config['data']['corpus']['n_talkers'] = 1
config['data']['corpus']['root'] = '/om/user/imgriff/datasets/dataset_word_speaker_noise/JSIN_all_v3/subsets/' # New path to JSIN dataset


snr=0
audio_transforms = at.AudioCompose([
                at.AudioToTensor(),
                at.CombineWithRandomDBSNR(low_snr=-snr, high_snr=snr), 
                at.RMSNormalizeForegroundAndBackground(rms_level=0.02),  # 0.02 is the default for CV-based models 
                at.UnsqueezeAudio(dim=0),
                at.DuplicateChannel() # only need to copy channels here
        ])  


# set up label re-mapping from SWC to CV labels 
word_and_speaker_encodings = pickle.load( open( "/om2/user/imgriff/projects/Auditory-Attention/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
# key is int, val is word
wsn_class_map = word_and_speaker_encodings['word_idx_to_word']
# key is word, val is int
cv_class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 
# map wsn class int key to cv class int value 
class_remap = {ix:(cv_class_map[word] if word in cv_class_map else -1) for ix, word in wsn_class_map.items()}

# def upsample op
upsample = T.Resample(20_000, sr,
                    lowpass_filter_width=64,
                    rolloff=0.9475937167399596,
                    beta=14.769656459379492,
                    resampling_method='kaiser_window',
                    dtype=torch.float32)
                            
# get dataset
dataset = jsinV3_attn_tracking_multi_talker_background(**config['data']['corpus'],
                                            mode='val',
                                            transform=None,
                                            demo=True)

def collate_fn(batch):
    #apply transforsms to batch
    cues = []
    foregrounds = []
    backgrounds = []
    mixtures = []
    labels = []
    for (fg, bg, cue, label) in batch:
        cue = audio_transforms(upsample(torch.from_numpy(cue)).squeeze(), None)[0]
        cues.append(cue)
        fg = upsample(torch.from_numpy(fg)).squeeze()
        bg = upsample(torch.from_numpy(bg)).squeeze()
        foregrounds.append(audio_transforms(fg, None)[0])
        backgrounds.append(audio_transforms(bg, None)[0])
        mixtures.append(audio_transforms(fg, bg)[0])
        labels.append(class_remap[label])

    cues = torch.stack(cues)
    foregrounds = torch.stack(foregrounds)
    backgrounds = torch.stack(backgrounds)
    mixtures = torch.stack(mixtures)
    labels = torch.tensor(labels).type(torch.LongTensor)
    return cues, foregrounds, backgrounds, mixtures, labels

dataloader = torch.utils.data.DataLoader(
            dataset,
            batch_size=1,
            num_workers=1,
            collate_fn=collate_fn
        )

1


In [6]:
list(model.model.model_dict.named_children() )

[('norm_coch_rep',
  LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)),
 ('attn0', SimpleAttentionalGain()),
 ('conv_block_0',
  Sequential(
    (0): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
    (1): Conv2d(2, 32, kernel_size=(2, 34), stride=(1, 1), bias=False)
    (2): ReLU()
  )),
 ('hann_pool_0', HannPooling2d()),
 ('attn1', SimpleAttentionalGain()),
 ('conv_block_1',
  Sequential(
    (0): LayerNorm((32, 20, 4992), eps=1e-05, elementwise_affine=True)
    (1): Conv2d(32, 64, kernel_size=(2, 14), stride=(1, 1), bias=False)
    (2): ReLU()
  )),
 ('hann_pool_1', HannPooling2d()),
 ('attn2', SimpleAttentionalGain()),
 ('conv_block_2',
  Sequential(
    (0): LayerNorm((64, 10, 1245), eps=1e-05, elementwise_affine=True)
    (1): Conv2d(64, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 0), bias=False)
    (2): ReLU()
  )),
 ('hann_pool_2', HannPooling2d()),
 ('attn3', SimpleAttentionalGain()),
 ('conv_block_3',
  Sequential(
    (0): LayerNorm((256,

In [7]:
## updates for v08 arch - just get all the modules 

conv_modules = {name:module for name, module in model.model._orig_mod.model_dict.named_children()} # if 'conv' in name or 'relu' in name or 'norm' in name or 'attn' in name  or 'p'}
# add relu fc 
relu_fc = model.model._orig_mod.relufc
modules = {**conv_modules, **{'relufc': relu_fc}}

In [8]:
# init array to store activations
activations = {}

def get_activation(name):
    def hook(model, input, output):
        # print(name)
        if name in activations:
            activations[name] = torch.cat((activations[name], output.detach()), dim=0)
        else:
            activations[name] = output.detach()
    return hook

# register hooks 
for name, module in modules.items():
    if 'conv' in name:
        module[0].register_forward_hook(get_activation(f"{name}_ln")) # [0] is layer norm 
        module[2].register_forward_hook(get_activation(f"{name}_relu")) # [2] is relu 
    else:
        module.register_forward_hook(get_activation(name)) 
    # lists to save acts in per-layer

# send model to gpu 
model = model.eval().cuda()


In [9]:
### Create module and forward hook that applys attnetional gains to activations
## Here we create an nn module that performs the same operations as SimpleAttentionalGain but only on the cue activations. 
## The parameters for each new module are taken from the existing checkpoint for each module in  model.attn_modules 

class AttentionalGains(nn.Module):
    def __init__(self, slope, bias, threshold):
        super(AttentionalGains, self).__init__()
        self.slope = slope.item()
        self.bias = bias.item()
        self.threshold = threshold.item()
        
    def forward(self, cue):
        cue = cue.mean(axis=-1,keepdim=True)
        # apply threshold shift
        cue = cue - self.threshold
        # apply slope
        cue = cue * self.slope
        # apply sigmoid & bias
        gain = self.bias + (1-self.bias) * torch.sigmoid(cue)
        return gain 
    
## Init gain modules per layer. 
gain_modules = {name:module for name,module in model.model._orig_mod.model_dict.items() if 'attn' in name}
gain_functions = {}


for name, module in gain_modules.items():
    gain_functions[name] = AttentionalGains(module.slope, module.bias, module.threshold)


In [10]:
gain_functions

{'attn0': AttentionalGains(),
 'attn1': AttentionalGains(),
 'attn2': AttentionalGains(),
 'attn3': AttentionalGains(),
 'attn4': AttentionalGains(),
 'attn5': AttentionalGains(),
 'attn6': AttentionalGains(),
 'attnfc': AttentionalGains()}

In [11]:
## Map pool layers to gain functions 
pool_layers = [name for name in modules.keys() if "pool" in name]
n_pool_layers = len(pool_layers)
pool_to_gain_map = {}
pool_to_gain_map['cochleagram'] = 'attn0'
for ix, layer in enumerate(pool_layers):
    if str(n_pool_layers - 1) in layer:
        pool_to_gain_map[layer] = 'attnfc'
    else:
        pool_to_gain_map[layer] = f"attn{ix+1}"
pool_to_gain_map

{'cochleagram': 'attn0',
 'hann_pool_0': 'attn1',
 'hann_pool_1': 'attn2',
 'hann_pool_2': 'attn3',
 'hann_pool_3': 'attn4',
 'hann_pool_4': 'attn5',
 'hann_pool_5': 'attn6',
 'hann_pool_6': 'attnfc'}

In [12]:
import h5py
from pathlib import Path
from scipy import stats 

In [13]:
# make map of module keys to gain keys 
len(modules.keys()), len(gain_modules.keys())

(24, 8)

In [14]:
# activations['conv_block_0_hannpool']

In [15]:
n_activations = 1 
# get activations 

# write acts to h5 
# outname = Path(f'binaural_model_attn_stage_reps/{model_name}/{model_name}_model_activations_0dB_w_cues_v2.h5')
# outname.parent.mkdir(parents=True, exist_ok=True)
# outname = 'test_act_outs.h5'
# with h5py.File(outname, 'w') as f:

with torch.no_grad():
    for ix, batch in tqdm(enumerate(dataloader),  total = n_activations):
        fg_cue, foreground, background, mixture, fg_target = batch

        # send to device
        foreground, background, mixture = foreground.cuda(), background.cuda(), mixture.cuda()
        fg_cue =  fg_cue.cuda()

        # convert to cochleagrams
        fg_cue, mixture = coch_gram(fg_cue, mixture)
        foreground, background = coch_gram(foreground, background)

        # if ix == 0:
        #         f.create_dataset('cochleagram_cue',shape=[n_activations, mixture.view(-1).shape[0]], dtype=np.float32)
        #         f.create_dataset('cochleagram_mixture', shape=[n_activations, mixture.view(-1).shape[0]], dtype=np.float32)
        #         f.create_dataset('cochleagram_fg', shape=[n_activations, mixture.view(-1).shape[0]], dtype=np.float32)
        #         f.create_dataset('cochleagram_bg', shape=[n_activations, mixture.view(-1).shape[0]], dtype=np.float32)
        # f['cochleagram_cue'][ix] = fg_cue.view(-1).cpu().numpy()
        # f['cochleagram_mixture'][ix] = mixture.view(-1).cpu().numpy()
        # f['cochleagram_fg'][ix] = foreground.view(-1).cpu().numpy()
        # f['cochleagram_bg'][ix] = background.view(-1).cpu().numpy()


        # run mixture
        activations = {}

        pred = model(fg_cue, mixture, None)
        for layer, acts in activations.items():
            print(layer)
            if 'relufc' in layer or 'attn' in layer:
                    mixture_acts = acts
                    print(acts.shape)
            else:
                cue_acts, mixture_acts = acts
                if 'pool' in layer:
                    gain_fn_name = pool_to_gain_map[layer]
                    gain_fn = gain_functions[gain_fn_name]
                    gains = gain_fn(cue_acts)
        break
      
        # if ix == n_activations-1:
        #     break
      

  0%|          | 0/1 [00:00<?, ?it/s]

[2024-06-27 12:49:58,456] [0/0] torch._dynamo.output_graph: [WARNING] nn.Module forward/_pre hooks are only partially supported, and were detected in your model. In particular, if you do not change/remove hooks after calling .compile(), you can disregard this warning, and otherwise you may need to set torch._dynamo.config.skip_nnmodule_hook_guards=False to ensure recompiling after changing hooks.See https://pytorch.org/docs/master/compile/nn-module.html for more information and limitations. 
  0%|          | 0/1 [00:14<?, ?it/s]

norm_coch_rep
attn0
torch.Size([1, 2, 40, 20000])
conv_block_0_ln
conv_block_0_relu
hann_pool_0
attn1
torch.Size([1, 32, 20, 4992])
conv_block_1_ln
conv_block_1_relu
hann_pool_1
attn2
torch.Size([1, 64, 10, 1245])
conv_block_2_ln
conv_block_2_relu
hann_pool_2
attn3
torch.Size([1, 256, 10, 249])
conv_block_3_ln
conv_block_3_relu
hann_pool_3
attn4
torch.Size([1, 512, 10, 62])
conv_block_4_ln
conv_block_4_relu
hann_pool_4
attn5
torch.Size([1, 512, 9, 57])
conv_block_5_ln
conv_block_5_relu
hann_pool_5
attn6
torch.Size([1, 512, 9, 53])
conv_block_6_ln
conv_block_6_relu
hann_pool_6
attnfc
torch.Size([1, 512, 5, 12])
relufc
torch.Size([1, 512])


In [16]:
gain_fn_name

'attnfc'

In [22]:
activations['hann_pool_0'].shape

torch.Size([2, 32, 20, 4992])

In [28]:
activations['attn1'].shape


torch.Size([1, 32, 20, 4992])

In [25]:
### see if manual application of gains gives same results as attended 

attended = activations['attn1']

# get gains from cue 
cue_acts, mixture = activations['hann_pool_0'] # 0 is cue, 1 is mixture 
gains = gain_functions['attn1'](cue_acts)
attended_manual = mixture * gains
## Manual gain application should be the same as attended - it is close enough to numerical precision. 
# Can save gains this way for old models. Will make gains own op in future models and just with a normal hook.
print(torch.allclose(attended, attended_manual, atol=1e-128))

True


True

In [ ]:
out_name =  Path(f'binaural_model_attn_stage_reps/{model_name}/{model_name}_layer_shape_dict.pkl')

shape_dict = {key:val.shape[1:] for key, val in activations.items()}
shape_dict['cochleagram'] = fg_cue.shape[1:]
print(shape_dict)
with open(out_name, 'wb') as f:
    pickle.dump(shape_dict, f)

{'conv_block_0': torch.Size([32, 40, 20000]), 'conv_block_1': torch.Size([64, 20, 5000]), 'conv_block_2': torch.Size([256, 10, 1250]), 'conv_block_3': torch.Size([512, 10, 250]), 'conv_block_4': torch.Size([512, 10, 63]), 'conv_block_5': torch.Size([512, 10, 63]), 'conv_block_6': torch.Size([512, 10, 63]), 'relufc': torch.Size([512]), 'cochleagram': torch.Size([2, 40, 20000])}


In [ ]:
out_name

PosixPath('binaural_model_attn_stage_reps/word_task_half_co_loc_v07/word_task_half_co_loc_v07_layer_shape_dict.pkl')

In [39]:
### Look at ouput file 
h5_fn = "binaural_unit_activations/word_task_half_co_loc_v08_gender_bal_4M_sanity/word_task_half_co_loc_v08_gender_bal_4M_sanity_model_activations_0dB.h5"
h5 = h5py.File(h5_fn, 'r')
print(list(h5.keys()))
print(np.allclose(h5['target_f0'][:100], h5['target_f0'][200:300]))
h5.close()

['attncoch_gains', 'cochleagram_cue', 'cochleagram_fg', 'cue_f0', 'layer_names', 'target_f0', 'target_loc']
True


True